# NEXUS — train the dual-sensor mutation→cell model on human PPIs

NEXUS scores a mutation with **two orthogonal structural sensors** and combines them into a graded enzyme activity:

- **Intrinsic** — does the protein still *fold*? (folding ΔΔG → `folded_fraction`)  ·  fixed, blind-validated model
- **Extrinsic** — can it still *dock* with partners? (interface ΔΔG + dMaSIF geodesic surface)  ·  **the trainable part**
- **Regulatory sign** — is the interface an ON or OFF switch? (breaks an *inhibitory* brake → **GOF**)  ·  annotation

The **trainable component is the dMaSIF surface model**, which keeps improving with more complexes (we measured the
gain *grow* with data). This notebook fetches human PPI complexes, trains dMaSIF on their interfaces (held out by
complex), and runs the full sensor stack end-to-end.

### How many human PPIs, and where to get them

| Layer | Count | Source |
|---|---|---|
| Binary PPIs (high-quality) | ~53,000 | HuRI (interactome-atlas.org) |
| All curated interactions | ~0.6–1M | BioGRID, IntAct, STRING (physical) |
| **PPIs with a 3D complex structure** ← NEXUS needs this | **a few thousand** | PDB, Interactome3D, **SKEMPI** (used here) |
| Monomer structures (intrinsic sensor) | ~all 20,000 | AlphaFold DB |
| Regulatory sign (GOF) | — | UniProt, SIGNOR |

The **binding sensor needs complex structures** — the real bottleneck. We train on SKEMPI's ~345 measured complexes
here; to go further, predict complexes with **AlphaFold-Multimer** and add them.


## 1 · GPU check (Runtime → Change runtime type → GPU for the scale-up run)


In [ ]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device, '|', torch.cuda.get_device_name(0) if device=='cuda' else 'CPU (fine for the 5-complex test)')


## 2 · Install the real tools
APBS + PDB2PQR (Poisson-Boltzmann electrostatics), scikit-image (marching-cubes surface), biopython, cobra.


In [ ]:
import sys, subprocess
print('installing APBS + PDB2PQR (apt) ...')
subprocess.run('apt-get -qq install -y apbs > /dev/null 2>&1', shell=True)
print('installing python deps (pip) ...')
subprocess.run(sys.executable+' -m pip -q install pdb2pqr scikit-image biopython cobra scikit-learn scipy 2>/dev/null', shell=True)
import shutil; print('apbs:', shutil.which('apbs'), '| pdb2pqr:', shutil.which('pdb2pqr'))


## 3 · Get the code (clone the repo branch)


In [ ]:
import os, sys
REPO='https://github.com/nikku03/cell.git'; BRANCH='claude/vectorize-gex-propensity-zp09w8'
if not os.path.isdir('/content/cell'):
    os.system(f'git clone -q -b {BRANCH} {REPO} /content/cell')
sys.path.insert(0, '/content/cell/colab')
print('modules:', 'ok' if os.path.isdir('/content/cell/colab') else 'MISSING (private repo? add a token to REPO url)')
print('ddg stability model:', os.path.exists('/content/cell/outputs/orphan/ddg_model.pkl'))


## 4 · Quick working-check — 5 complexes (~1–2 min on CPU)
Verifies the whole pipeline: fetch → surface+APBS → train dMaSIF (held-out) → run the sensor stack.


In [ ]:
import nexus_train, importlib; importlib.reload(nexus_train)
PDBS = ['1BRS','1JCK','3SGB','1ACB','1AHW']   # start small
res = nexus_train.main(PDBS, cache='/content/pdb_cache', epochs=15, device=device)
print('\nWORKING-CHECK:', 'PASS' if res.get('working_check') else 'FAIL',
      '| held-out dMaSIF AUC', res.get('dmasif_heldout_auc'))


## 5 · Scale up — train on ALL ~345 SKEMPI complexes (use GPU)
Downloads SKEMPI, extracts every complex PDB id, and trains dMaSIF on all of them.
On a Colab **GPU** this is the version that actually improves the surface sensor. Expect ~30–60 min
(APBS is the bottleneck; it runs on CPU per complex, training on GPU).


In [ ]:
import urllib.request, csv, os
os.makedirs('/content', exist_ok=True)
SK='/content/skempi_v2.csv'
if not os.path.exists(SK):
    urllib.request.urlretrieve('https://life.bsc.es/pid/skempi2/database/download/skempi_v2.csv', SK)
rows = list(csv.reader(open(SK), delimiter=';'))[1:]
ALL = sorted({r[0].split('_')[0] for r in rows if r and r[0]})
print(len(ALL), 'SKEMPI complexes')
# NOTE: uncomment to run the full training (long). Start with a slice to gauge time:
# res = nexus_train.main(ALL[:40], cache='/content/pdb_cache', epochs=40, device=device)
# res = nexus_train.main(ALL,      cache='/content/pdb_cache', epochs=60, device=device)


## 6 · Use your own PPI list (HuRI / BioGRID / AlphaFold-Multimer)
Any list of **PDB ids of complexes** works. To go beyond solved structures, predict complexes with
AlphaFold-Multimer for HuRI pairs, save the `.pdb` files into the cache dir, and pass their ids.
The trained surface model is saved to `outputs/nexus_dmasif.pt` and the sensor report to `outputs/orphan/nexus_train.json`.


In [ ]:
# MYPPIS = ['4HFK','1DVF', ...]   # your complex PDB ids
# res = nexus_train.main(MYPPIS, cache='/content/pdb_cache', epochs=40, device=device)


## What you get — and the honest edges
- **Trained:** the dMaSIF geodesic **surface (extrinsic) sensor** — the part that scales with data.
- **Fixed:** the intrinsic **stability** node (blind-validated) and the **regulatory-sign** GOF layer (annotation).
- **Validated here:** held-out interface-discrimination AUC (dMaSIF) + the sensor stack running end-to-end.

**Honest limits (carried from the validation):** the binding sensor needs *complex* structures (thousands, not 20k —
the rest need AlphaFold-Multimer); the metabolic (FBA) consequence is a *demonstration*, not a validated phenotype
predictor (the far-field step doesn't compose); *neomorphic* GOF (a brand-new interface) is out of reach; and the
regulatory sign is high-precision but annotation-**recall-limited**.
